# 01 — EDA Titanic

**EDA (Exploratory Data Analysis)** = "kenalan" dengan data: isinya apa, mana yang kosong, kolom mana yang berhubungan dengan target.

Cara pakai:
- **Di laptop:** pilih kernel **Python (kaggle)** → jalankan sel dari atas.
- **Di GPU Colab (VS Code):** pilih kernel **Colab** → sel setup akan `git clone` repo, sel token akan minta token Kaggle.

Aturan main: sel di bagian **Eksplorasi** bebas kamu ubah. Kode yang mau dipakai lagi → minta Claude pindahkan ke `src/`.

In [ ]:
# === Sel SETUP — jalankan PERTAMA, dan jalankan ulang setelah Claude mengubah file .py ===
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/mrizkykhatami/titanic.git"   # ganti kalau nama repo berbeda

def _remote():
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
        return "kaggle"
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        return None

where = _remote()
if where:  # Kaggle / Colab: ambil kode terbaru dari GitHub
    ROOT = (Path("/kaggle/working") if where == "kaggle" else Path("/content")) / "titanic"
    if ROOT.exists():
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
else:      # Laptop: pakai folder repo ini langsung
    ROOT = Path.cwd().parent if Path.cwd().name in ("notebooks", "train") else Path.cwd()
    if ROOT.name == "kernels":
        ROOT = ROOT.parent

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("jalan di:", where or "local", "| ROOT =", ROOT)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# === Token Kaggle — hanya diperlukan di Colab (di laptop sudah ada ~/.kaggle/access_token) ===
import os
from getpass import getpass
if where == "colab" and not os.environ.get("KAGGLE_API_TOKEN"):
    os.environ["KAGGLE_API_TOKEN"] = getpass("Tempel Kaggle API token: ")  # tidak tampil & tidak tersimpan di notebook

## 1. Muat data

In [ ]:
from src.data import load
train, test = load()
train.head()

In [ ]:
# Berapa nilai kosong (NaN) per kolom?
train.isna().sum().sort_values(ascending=False)

## 2. Eksplorasi (bebas diubah)

In [ ]:
# Persentase selamat secara keseluruhan
train["Survived"].mean()

In [ ]:
# Persentase selamat per jenis kelamin & kelas tiket
train.pivot_table(index="Sex", columns="Pclass", values="Survived", aggfunc="mean").round(2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(data=train, x="Pclass", y="Survived", hue="Sex", ax=axes[0])
axes[0].set_title("Peluang selamat per kelas & gender")
sns.histplot(data=train, x="Age", hue="Survived", bins=30, ax=axes[1])
axes[1].set_title("Distribusi umur vs selamat")
plt.tight_layout()

## 3. Feature engineering (kode ada di `src/features.py`)

In [ ]:
from src.features import add_features
feat = add_features(train)
feat[["Name", "Title", "FamilySize", "IsAlone", "Deck", "Survived"]].head(10)

In [ ]:
feat.groupby("Title")["Survived"].agg(["mean", "count"]).round(2)

## 4. Cross-Validation (kode ada di `src/train.py`)

Model dilatih 5 kali (5-fold). Skor **CV** = seberapa akurat tebakan model pada data yang tidak pernah ia lihat.

In [ ]:
from src.train import run_cv
result = run_cv(model="hgb", n_splits=5, seed=42)
result

In [ ]:
# Bandingkan beberapa model sekaligus
import pandas as pd
pd.DataFrame([run_cv(model=m) for m in ["logreg", "rf", "hgb"]])[["name", "cv_accuracy", "fold_std"]]

## 5. Langkah berikutnya
- File `outputs/<nama>/submission.csv` siap dikirim ke Kaggle.
- Ide eksperimen: tambah feature baru di `src/features.py`, ganti hyperparameter di `configs/`, atau gabungkan beberapa model (ensemble).